# Start

In [1]:
import clip
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
from string import punctuation
import re

In [2]:
import sys
import os
import importlib.util

repo_path = "../vision-language-models-are-bows"    # from https://github.com/mertyg/vision-language-models-are-bows
sys.path.append(repo_path)  # Add to Python path

module_name = "dataset_zoo"  # The submodule to import
module_path = os.path.join(repo_path, module_name)

spec = importlib.util.find_spec(module_name)
if spec is None:
    raise ImportError(f"Module {module_name} not found in {repo_path}")

dataset_zoo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dataset_zoo)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load('ViT-B/32', device=device)

# VG-A and VG-R

In [ ]:
from dataset_zoo import VG_Relation, VG_Attribution

root_dir="../datasets/ARO"

vgr_dataset = VG_Relation(image_preprocess=preprocess, download=False, root_dir=root_dir)    # set download=True to download the dataset
vga_dataset = VG_Attribution(image_preprocess=preprocess, download=False, root_dir=root_dir)


In [8]:
def get_embeddings(model, dataset):
    model = model.eval()
    image_embeddings, text_embeddings_pos, text_embeddings_neg = [], [], []

    # iterate over the dataset
    for item in tqdm(dataset):
        image = item["image_options"][0].unsqueeze(0).to(device)
        pos_caption = item["caption_options"][1]
        neg_caption = item["caption_options"][0]

        # get embeddings
        with torch.no_grad():
            image_embedding = model.encode_image(image)
            pos_text_embedding = model.encode_text(clip.tokenize(pos_caption).to(device))
            neg_text_embedding = model.encode_text(clip.tokenize(neg_caption).to(device))

        image_embeddings.append(image_embedding)
        text_embeddings_pos.append(pos_text_embedding)
        text_embeddings_neg.append(neg_text_embedding)

    image_embeddings = torch.stack(image_embeddings)
    text_embeddings_pos = torch.stack(text_embeddings_pos)
    text_embeddings_neg = torch.stack(text_embeddings_neg)

    # normalize embeddings
    image_embeddings /= image_embeddings.norm(dim=-1, keepdim=True)
    text_embeddings_pos /= text_embeddings_pos.norm(dim=-1, keepdim=True)
    text_embeddings_neg /= text_embeddings_neg.norm(dim=-1, keepdim=True)

    return image_embeddings, text_embeddings_pos, text_embeddings_neg

In [ ]:
vga_image_embeddings, vga_text_embeddings_pos, vga_text_embeddings_neg = get_embeddings(model, vga_dataset)

In [ ]:
vgr_image_embeddings, vgr_text_embeddings_pos, vgr_text_embeddings_neg = get_embeddings(model, vgr_dataset)

In [5]:
# # VGA
# # save the embeddings
# torch.save(vga_image_embeddings, "../cache/vga_image_emb.pt")
# torch.save(vga_text_embeddings_pos, "../cache/vga_text_emb_pos.pt")
# torch.save(vga_text_embeddings_neg, "../cache/vga_text_emb_neg.pt")

# # load the embeddings
# vga_image_embeddings = torch.load("../cache/vga_image_emb.pt")
# vga_text_embeddings_pos = torch.load("../cache/vga_text_emb_pos.pt")
# vga_text_embeddings_neg = torch.load("../cache/vga_text_emb_neg.pt")

# # VGR
# # save the embeddings
# torch.save(vgr_image_embeddings, "../cache/vgr_image_emb.pt")
# torch.save(vgr_text_embeddings_pos, "../cache/vgr_text_emb_pos.pt")
# torch.save(vgr_text_embeddings_neg, "../cache/vgr_text_emb_neg.pt")

# # load the embeddings
# vgr_image_embeddings = torch.load("../cache/vgr_image_emb.pt")
# vgr_text_embeddings_pos = torch.load("../cache/vgr_text_emb_pos.pt")
# vgr_text_embeddings_neg = torch.load("../cache/vgr_text_emb_neg.pt")

In [11]:
def calculate_accuracy(dataset, image_embeddings, text_embeddings_pos, text_embeddings_neg):

    similarity_pos = (image_embeddings.squeeze() * text_embeddings_pos.squeeze()).sum(dim=-1)   # similarity between image and positive text
    similarity_neg = (image_embeddings.squeeze() * text_embeddings_neg.squeeze()).sum(dim=-1)   # similarity between image and negative text

    # concatenate similarity_pos and similarity_neg
    scores = torch.cat((similarity_neg.unsqueeze(-1), similarity_pos.unsqueeze(-1)), dim=-1).unsqueeze(1)
    evaluation = dataset.evaluate_scores(scores.cpu().numpy())

    accuracies = []
    for data in evaluation:
        acc = data['Accuracy']
        accuracies.append(acc)

    print(np.mean(accuracies))

In [ ]:
calculate_accuracy(vga_dataset, vga_image_embeddings, vga_text_embeddings_pos, vga_text_embeddings_neg)
calculate_accuracy(vgr_dataset, vgr_image_embeddings, vgr_text_embeddings_pos, vgr_text_embeddings_neg)

In [ ]:
from learning_alignment import CLIPAlignment

# load the alignment model
model = CLIPAlignment(vga_image_embeddings.shape[-1]).to(device)
model.load_state_dict(torch.load('../cache/coco_alignment_hnb.pt'))
model.eval()

# get the aligned embeddings
with torch.no_grad():
    vga_text_embeddings_pos_a = model(vga_text_embeddings_pos.float())
    vga_text_embeddings_neg_a = model(vga_text_embeddings_neg.float())
    vgr_text_embeddings_pos_a = model(vgr_text_embeddings_pos.float())
    vgr_text_embeddings_neg_a = model(vgr_text_embeddings_neg.float())

calculate_accuracy(vga_dataset, vga_image_embeddings, vga_text_embeddings_pos_a, vga_text_embeddings_neg_a)
calculate_accuracy(vgr_dataset, vgr_image_embeddings, vgr_text_embeddings_pos_a, vgr_text_embeddings_neg_a)

# COCO Order

In [ ]:
from dataset_zoo import COCO_Order

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, preprocess = clip.load('ViT-B/32', device=device)

root_dir="../datasets/ARO"
coco_order_dataset = COCO_Order(image_preprocess=preprocess, download=False, root_dir=root_dir)    # set download=True to download the dataset

In [ ]:
corr = 0
all_image_embeddings = []
all_text_embeddings = []

# iterate over the dataset
for item in tqdm(coco_order_dataset):
    image = item["image_options"][0].unsqueeze(0).to(device)
    caption_options = item["caption_options"]

    # get embeddings
    with torch.no_grad():
        image_embedding = model.encode_image(image)
        text_embeddings = model.encode_text(clip.tokenize(caption_options).to(device))

        image_embedding = image_embedding / image_embedding.norm(dim=-1, keepdim=True)
        text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)

        all_image_embeddings.append(image_embedding)
        all_text_embeddings.append(text_embeddings)

        # calculate similarity
        similarity = image_embedding @ text_embeddings.T
        pred = similarity.argmax(dim=1).item()

        if pred == 0:    # correct caption is the first one
            corr += 1

print(f"Accuracy: {corr/len(coco_order_dataset)}")

In [19]:
# # save the embeddings
# torch.save(all_image_embeddings, "../cache/coco_order_image_emb.pt")
# torch.save(all_text_embeddings, "../cache/coco_order_text_emb.pt")

# # load the embeddings
# all_image_embeddings = torch.load("../cache/coco_order_image_emb.pt")
# all_text_embeddings = torch.load("../cache/coco_order_text_emb.pt")

In [ ]:
# load the alignment model
model = CLIPAlignment(all_image_embeddings.shape[-1]).to(device)
model.load_state_dict(torch.load('coco_cache/coco_alignment_hnb.pt'))
model.eval()

corr = 0
for i in range(len(all_image_embeddings)):
    image_embedding = all_image_embeddings[i].float()
    text_embeddings = all_text_embeddings[i].float()

    # get the aligned embeddings
    text_embeddings = model(text_embeddings)

    similarity = image_embedding @ text_embeddings.T
    pred = similarity.argmax(dim=1).item()

    if pred == 0:   # correct caption is the first one
        corr += 1

print(f"Accuracy: {corr/len(coco_order_dataset)}")

# Flickr30k

In [ ]:
from dataset_zoo import Flickr30k_Order

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load('ViT-B/32', device=device)

root_dir="../datasets/ARO"
flickr_order_dataset = Flickr30k_Order(image_preprocess=preprocess, download=True, split='test', root_dir=root_dir)

In [ ]:
corr = 0
all_image_embeddings = []
all_text_embeddings = []

# iterate over the dataset
for item in tqdm(flickr_order_dataset):
    image = item["image_options"][0].unsqueeze(0).to(device)
    caption_options = item["caption_options"]

    with torch.no_grad():
        # get embeddings
        image_embedding = model.encode_image(image)
        text_embeddings = model.encode_text(clip.tokenize(caption_options).to(device))

        # normalize embeddings
        image_embedding = image_embedding / image_embedding.norm(dim=-1, keepdim=True)
        text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)

        all_image_embeddings.append(image_embedding)
        all_text_embeddings.append(text_embeddings)

        similarity = image_embedding @ text_embeddings.T
        pred = similarity.argmax(dim=1).item()

        if pred == 0:
            corr += 1

print(f"Accuracy: {corr/len(flickr_order_dataset)}")

In [23]:
# save the embeddings
# torch.save(all_image_embeddings, "../cache/flickr_order_image_emb.pt")
# torch.save(all_text_embeddings, "../cache/flickr_order_text_emb.pt")

# all_image_embeddings = torch.load("../cache/flickr_order_image_emb.pt")
# all_text_embeddings = torch.load("../cache/flickr_order_text_emb.pt")

In [ ]:
model = CLIPAlignment(all_image_embeddings.shape[-1]).to(device)
model.load_state_dict(torch.load('../cache/coco_alignment_hnb.pt'))
model.eval()

corr = 0
for i in range(len(all_image_embeddings)):
    image_embedding = all_image_embeddings[i].float()
    text_embeddings = all_text_embeddings[i].float()

    text_embeddings = model(text_embeddings)

    similarity = image_embedding @ text_embeddings.T
    pred = similarity.argmax(dim=1).item()

    if pred == 0:
        corr += 1

print(f"Accuracy: {corr/len(flickr_order_dataset)}")
